In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
import pandas as pd
df = pd.read_excel(file_name)


# your input dataframe = df

# clean size (important)
df['Size'] = df['Size'].astype(str).str.strip().str.upper()

# handle 2X,3X,4X → 2XL,3XL,4XL
size_map = {'2X':'2XL','3X':'3XL','4X':'4XL'}
df['Size'] = df['Size'].replace(size_map)

# Convert 'Line Qty' to numeric, coercing errors to NaN and then fill NaN with 0
df['Line Qty'] = pd.to_numeric(df['Line Qty'], errors='coerce').fillna(0)

# pivot (this is the main step)
pivot = df.pivot_table(
    index=['Style Number','PO Number','Color'],
    columns='Size',
    values='Line Qty',
    aggfunc='sum',
    fill_value=0
).reset_index()

# ensure all size columns exist
size_cols = ['XS','S','M','L','XL','2XL','3XL','4XL']
for s in size_cols:
    if s not in pivot.columns:
        pivot[s] = 0

# reorder columns
pivot = pivot[['Style Number','PO Number','Color'] + size_cols]

print(pivot)

output_file = "FINAL_OUTPUT_SORTED.xlsx"
pivot.to_excel(output_file, index=False)

files.download(output_file)

Saving SIZE_WISE_TOTAL (5).xlsx to SIZE_WISE_TOTAL (5) (1).xlsx
Size  Style Number  PO Number Color  XS   S   M   L  XL  2XL  3XL  4XL
0             1215    1738230   ASH   0   0   0  24   0    0    0    0
1             1215    1738230    NY  24   0  24   0   0    0    0    0
2             1215   10007049    NY   0   0   0  48   0    0    0    0
3             1215   10007049    RY   0   0   0   0   0   48    0    0
4             1215   10007081    BK   0   0   0   0   0    0   36    0
..             ...        ...   ...  ..  ..  ..  ..  ..  ...  ...  ...
72            5577    1738643    CH  24  24   0   0  24    0    0    0
73            5577    1735988    CH   0   0   0   0   0    0    0   12
74            5577    1736475    CH   0   0  24  24   0    0    0    0
75            5578    1738230    NY   0   0   0  24   0    0    0    0
76            5578    1738643    BK  24   0   0   0   0   48    0    0

[77 rows x 11 columns]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 2.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from google.colab import files
import IPython.display as display
import openpyxl
import os

print("Please upload your main Excel file (e.g., 'Container Plan & WIP.xlsx').")
# Upload the main file
uploaded_main_file = files.upload()

# Get the name of the uploaded main file
main_file_name = list(uploaded_main_file.keys())[0]

# Define keywords for flexible header row detection, focused on the exact headers provided by the user
# All keywords converted to uppercase for case-insensitive matching
header_detection_keywords = [
    "ORD #", "PO #", "DUE DATE", "STYLE #", "COLOR", "XS", "S", "M", "L", "XL",
    "2XL", "3XL", "4XL", "TOTAL", "ESTIMATED DATE"
]

# Function to find the header row by matching keywords
def find_header_row(df_to_check, detection_keywords, min_matches=5):
    for i, row in df_to_check.iterrows():
        row_values = [str(cell).strip().upper() for cell in row.values if pd.notna(cell)]
        match_count = sum(1 for keyword in detection_keywords if any(keyword.upper() in val for val in row_values))
        # Set a threshold for min_matches based on the expected number of headers
        if match_count >= min_matches:
            return i
    return None

# Helper function for robust column name standardization
def standardize_col_name(col_name):
    # Convert to string, strip whitespace, convert to uppercase
    standardized = str(col_name).strip().upper()
    # Replace common variations of '#' to '# ' if not already there, then collapse spaces
    if '#' in standardized and '# ' not in standardized:
        standardized = standardized.replace('#', ' #')
    # Collapse multiple spaces into a single space and strip again
    standardized = ' '.join(standardized.split())
    return standardized


print(f"Successfully uploaded '{main_file_name}'. Now attempting to extract 'COLOR' column from Sheets.")

# Read Sheet 2 (0-indexed, so sheet_name=1) without assuming header to find the header row
df_raw_sheet2 = pd.read_excel(main_file_name, sheet_name=1, header=None)
df_raw_sheet3 = pd.read_excel(main_file_name, sheet_name=2, header=None)
df_raw_sheet4 = pd.read_excel(main_file_name, sheet_name=3, header=None)
# Find the header row in Sheet 2
# Using min_matches=5 to ensure a robust match with the provided specific headers
header_row_index_sheet2 = find_header_row(df_raw_sheet2, header_detection_keywords, min_matches=5)
header_row_index_sheet3 = find_header_row(df_raw_sheet3, header_detection_keywords, min_matches=5)
header_row_index_sheet4 = find_header_row(df_raw_sheet4, header_detection_keywords, min_matches=5)

# Expanded color_mapping dictionary for better standardization
color_mapping = {
    'BK': 'BLACK',
    'WH': 'WHITE',
    'GY': 'GREY',
    'RD': 'RED',
    'BL': 'BLUE',
    'GN': 'GREEN',
    'YL': 'YELLOW',
    'OR': 'ORANGE',
    'PR': 'PURPLE',
    'BN': 'BROWN',
    'OX': 'OXFORD',
    'RY': 'ROYAL',
    'NV': 'NAVY',
    'CR': 'CHARCOAL',
    'PK': 'PINK',
    'TQ': 'TURQUOISE',
    'BG': 'BURGUNDY',
    'GR': 'GRAPHITE',
    'OL': 'OLIVE',
    'LM': 'LIME',
    'MA': 'MAROON',
    'HTR': 'HEATHER',
    'PU': 'PURPLE',
    'FS': 'FOREST',
    'BO': 'BURNT ORANGE',
    'NY': 'NAVY',
    # New abbreviations/full names identified from unique list
    'CH': 'CHARCOAL',
    'KY': 'KELLY GREEN',
    'SFYW': 'SAFETY YELLOW',
    'VG': 'VEGAS GOLD',
    'GD': 'GOLD',
    'CB': 'COLUMBIA BLUE',
    'BR': 'BROWN',
    'MI': 'MILITARY',
    'ASH': 'ASH',
    'CARDINAL': 'CARDINAL',
    # Direct mappings for composite-like single names (if not split by '/')
    'GDHT': 'GOLD HEATHER',
    'GRHT': 'GREY HEATHER',
    'RDHT': 'RED HEATHER',
    'PUHT': 'PURPLE HEATHER',
    'RYHT': 'ROYAL HEATHER',
    'CBHT': 'COLUMBIA BLUE HEATHER',
    'FSHT': 'FOREST HEATHER',
    'MAHT': 'MAROON HEATHER',
    'BOHT': 'BURNT ORANGE HEATHER',
    # Also include full names to ensure they map to themselves explicitly
    'BLACK': 'BLACK',
    'WHITE': 'WHITE',
    'GREY': 'GREY',
    'RED': 'RED',
    'BLUE': 'BLUE',
    'GREEN': 'GREEN',
    'YELLOW': 'YELLOW',
    'ORANGE': 'ORANGE',
    'BROWN': 'BROWN',
    'OXFORD': 'OXFORD',
    'ROYAL': 'ROYAL',
    'NAVY': 'NAVY',
    'CHARCOAL': 'CHARCOAL',
    'PINK': 'PINK',
    'TURQUOISE': 'TURQUOISE',
    'BURGUNDY': 'BURGUNDY',
    'GRAPHITE': 'GRAPHITE',
    'OLIVE': 'OLIVE',
    'LIME': 'LIME',
    'MAROON': 'MAROON',
    'HEATHER': 'HEATHER',
    'FOREST': 'FOREST',
    'BURNT ORANGE': 'BURNT ORANGE',
    'COLUMBIA BLUE': 'COLUMBIA BLUE',
    'KELLY GREEN': 'KELLY GREEN',
    'SAFETY YELLOW': 'SAFETY YELLOW',
    'VEGAS GOLD': 'VEGAS GOLD',
    'GOLD': 'GOLD',
    'MILITARY': 'MILITARY'
}

# Define a function to apply the mapping to each component of a compound color string
def map_compound_color(color_string, mapping):
    if pd.isna(color_string) or color_string == '':
        return 'UNKNOWN' # Treat NaN or empty strings as 'UNKNOWN'

    original_parts = str(color_string).split('/')
    processed_parts = []
    for part in original_parts:
        part_upper = part.strip().upper()
        # Handle FSG prefix removal and mapping
        if part_upper.startswith('FSG '):
            sub_part = part_upper[4:].strip() # Remove 'FSG ' and strip
            if sub_part in mapping:
                processed_parts.append(mapping[sub_part]) # Remove 'FSG ' and map
            else:
                processed_parts.append(sub_part) # Remove 'FSG ' but keep sub_part if not mapped
        elif part_upper in mapping:
            processed_parts.append(mapping[part_upper])
        else:
            processed_parts.append(part_upper) # Ensure part is uppercase before comparison/inclusion

    # Remove 'WHITE' if it's in a compound color (i.e., if there's more than one processed part)
    final_parts = list(processed_parts)
    if len(final_parts) ==3 and final_parts[2] == 'WHITE':
        final_parts.pop() # Remove 'WHITE'
        final_parts.append('1')
        if not final_parts: # If all components were removed (e.g., only 'WHITE' in a compound), return 'UNKNOWN'
            return 'UNKNOWN'
    else: # Not a compound color, or only two part
        final_parts = processed_parts
    m='/'.join(final_parts).strip('1')
    return m


all_sheets_data = {}
# Corrected loop to use sheet index instead of hardcoded string names
sheet_info_original = [
    ("Sheet 2", 1, header_row_index_sheet2),
    ("Sheet 3", 2, header_row_index_sheet3),
    ("Sheet 4", 3, header_row_index_sheet4)
]

for descriptive_sheet_name, sheet_index, header_row_index in sheet_info_original:
    if header_row_index is not None:
        # Use sheet_index for pd.read_excel to correctly identify the sheet
        df_sheet = pd.read_excel(main_file_name, sheet_name=sheet_index, header=header_row_index)
        if (df_sheet.any() == "GRAND TOTAL").any().any():
          continue
        # Standardize column names for easier access: make them uppercase, strip spaces, replace multiple spaces.
        df_sheet.columns = [standardize_col_name(col) for col in df_sheet.columns]

        # Check only for 'COLOR' column and extract color information
        if 'COLOR' in df_sheet.columns:
            # Apply normalization to the 'COLOR' column
            df_sheet['COLOR'] = df_sheet['COLOR'].astype(str).apply(lambda x: map_compound_color(x, color_mapping))
            color_col_content = df_sheet['COLOR'].astype(str).str.strip()

        # Store the processed dataframe in the dictionary
        all_sheets_data[descriptive_sheet_name] = df_sheet

# Concatenate all dataframes into a single dataframe
df_combined = pd.concat(all_sheets_data.values(), ignore_index=True)

# Ensure PO # and STYLE # are of appropriate numeric types for comparison
# Convert 'PO #' and 'STYLE #' columns in df_combined to numeric, coercing errors
df_combined['PO #'] = pd.to_numeric(df_combined['PO #'], errors='coerce')
df_combined['STYLE #'] = pd.to_numeric(df_combined['STYLE #'], errors='coerce')

# *** ADDED: Take a snapshot of df_combined BEFORE updates ***
df_combined_before_update = df_combined.copy()

print("\n--- Now, please upload the Excel or CSV file containing your updates. ---")
print("It should have 'PO #', 'STYLE #', 'COLOR' columns and all size columns (XS, S, M, L, XL, 2XL, 3XL, 4XL).")

uploaded_update_file = files.upload()
update_file_name = list(uploaded_update_file.keys())[0]

# Read the update file
if update_file_name.endswith('.csv'):
    df_updates = pd.read_csv(update_file_name)
elif update_file_name.endswith(('.xls', '.xlsx')):
    df_updates = pd.read_excel(update_file_name)
else:
    raise ValueError("Unsupported update file format. Please upload a .csv, .xls, or .xlsx file.")

# Standardize columns of the update DataFrame
df_updates.columns = [standardize_col_name(col) for col in df_updates.columns]

# Apply color mapping to the 'COLOR' column in df_updates
if 'COLOR' in df_updates.columns:
    df_updates['COLOR'] = df_updates['COLOR'].astype(str).apply(lambda x: map_compound_color(x, color_mapping))

# Ensure PO # and STYLE # in df_updates are numeric
df_updates['PO #'] = pd.to_numeric(df_updates['PO #'], errors='coerce')
df_updates['STYLE #'] = pd.to_numeric(df_updates['STYLE #'], errors='coerce')

# Identify the columns that represent sizes, make sure they are in the update file
size_columns = ['XS', 'S', 'M', 'L', 'XL', '2XL', '3XL', '4XL']
actual_size_cols_in_updates = [col for col in size_columns if col in df_updates.columns]

if not actual_size_cols_in_updates:
    print("Warning: No valid size columns (XS, S, M, L, XL, 2XL, 3XL, 4XL) found in the update file. No size updates will be applied.")

# Display update data head for confirmation
print(f"Successfully loaded update file '{update_file_name}'. First 5 rows:")
display.display(df_updates.head())

print("\n--- Debugging: Standardized Column Names ---")
print("df_combined columns:", df_combined.columns.tolist())
print("df_updates columns:", df_updates.columns.tolist())

# --- Apply updates to df_combined (Pandas DataFrame) and the openpyxl workbook ---

output_file_name_formatted = 'Updated_Container_Plan_Formatted.xlsx'

# List to keep track of indices from df_updates that were successfully applied
matched_update_indices = []

# Initialize sheet-specific information for inserting new rows
badger_ws = None
c2_ws = None
badger_header_row_index = None
c2_header_row_index = None
badger_col_name_to_excel_col_idx = {}
c2_col_name_to_excel_col_idx = {}
badger_excel_column_order = []
c2_excel_column_order = []

if main_file_name.endswith('.xls'):
    print("Error: openpyxl does not support the old .xls file format for preserving formatting.")
    print("Please convert your main Excel file to the .xlsx format and re-upload it to use the formatting preservation feature.")
else:
    # Load the existing workbook to preserve formatting
    workbook = openpyxl.load_workbook(main_file_name)

    # Populate sheet-specific information for Badger and C2 sheets (assuming Sheet 2 is Badger, Sheet 3 is C2)
    for descriptive_sheet_name, sheet_index, header_row_index_found in sheet_info_original:
        if header_row_index_found is not None and sheet_index < len(workbook.worksheets):
            ws = workbook.worksheets[sheet_index]
            current_col_name_to_excel_col_idx = {}
            current_excel_column_order = []

            # Build col_name_to_excel_col_idx and excel_column_order for the current sheet
            header_row_in_excel = ws[header_row_index_found + 1] # openpyxl rows are 1-indexed
            max_col_idx = 0
            for col_idx_openpyxl_1_based, cell in enumerate(header_row_in_excel, 1):
                if cell.value is not None:
                    excel_header_val_standardized = standardize_col_name(cell.value)
                    current_col_name_to_excel_col_idx[excel_header_val_standardized] = col_idx_openpyxl_1_based
                    if col_idx_openpyxl_1_based > max_col_idx:
                        max_col_idx = col_idx_openpyxl_1_based

            if max_col_idx > 0:
                current_excel_column_order = [None] * max_col_idx
                for col_name, col_idx in current_col_name_to_excel_col_idx.items():
                    current_excel_column_order[col_idx - 1] = col_name

            if descriptive_sheet_name == "Sheet 2": # Assuming Sheet 2 is "Badger"
                badger_ws = ws
                badger_header_row_index = header_row_index_found
                badger_col_name_to_excel_col_idx = current_col_name_to_excel_col_idx
                badger_excel_column_order = current_excel_column_order
            elif descriptive_sheet_name == "Sheet 3": # Assuming Sheet 3 is "C2"
                c2_ws = ws
                c2_header_row_index = header_row_index_found
                c2_col_name_to_excel_col_idx = current_col_name_to_excel_col_idx
                c2_excel_column_order = current_excel_column_order

    total_updates_applied = 0
    for idx, update_row in df_updates.iterrows():
        po_to_update = update_row.get('PO #')
        style_to_update = update_row.get('STYLE #')
        color_to_update = update_row.get('COLOR')

        if pd.isna(po_to_update) or pd.isna(style_to_update) or pd.isna(color_to_update):
            continue

        new_sizes_values = {col: update_row[col] for col in actual_size_cols_in_updates if pd.notna(update_row[col])}

        if not new_sizes_values:
            continue

        # --- Update df_combined (Pandas DataFrame) ---
        matching_rows_index_df_combined = df_combined[
            (df_combined['PO #'] == po_to_update) &
            (df_combined['STYLE #'] == style_to_update) &
            (df_combined['COLOR'].str.upper().fillna('') == str(color_to_update).upper())
        ].index

        if not matching_rows_index_df_combined.empty:
            for size_col, new_val in new_sizes_values.items():
                if size_col in df_combined.columns:
                    df_combined.loc[matching_rows_index_df_combined, size_col] = new_val
            total_updates_applied += 1
            matched_update_indices.append(idx)

            # --- Apply updates to the openpyxl workbook for existing rows ---
            for descriptive_sheet_name_orig, sheet_idx_orig, header_row_idx_orig in sheet_info_original:
                if descriptive_sheet_name_orig in all_sheets_data: # Check if this sheet was successfully processed
                    if sheet_idx_orig < len(workbook.worksheets):
                        ws = workbook.worksheets[sheet_idx_orig]

                        col_name_to_excel_col_idx = {}
                        standardized_df_columns = list(all_sheets_data[descriptive_sheet_name_orig].columns)

                        for col_idx_openpyxl_1_based, cell in enumerate(ws[header_row_idx_orig + 1], 1):
                            if cell.value is not None:
                                excel_header_val_standardized = standardize_col_name(cell.value)
                                if excel_header_val_standardized in standardized_df_columns:
                                    col_name_to_excel_col_idx[excel_header_val_standardized] = col_idx_openpyxl_1_based

                        # Find matching rows in this specific pandas DataFrame (df_current_sheet) for accurate row indexing
                        df_current_sheet_for_matching = all_sheets_data[descriptive_sheet_name_orig]
                        matching_rows_in_df_sheet = df_current_sheet_for_matching[
                            (df_current_sheet_for_matching['PO #'] == po_to_update) &
                            (df_current_sheet_for_matching['STYLE #'] == style_to_update) &
                            (df_current_sheet_for_matching['COLOR'].str.upper().fillna('') == str(color_to_update).upper())
                        ]

                        if not matching_rows_in_df_sheet.empty:
                            for df_row_index, _df_row in matching_rows_in_df_sheet.iterrows():
                                # The actual row number in Excel (1-indexed)
                                excel_row_num = df_row_index + header_row_idx_orig + 2

                                for size_col, new_val in new_sizes_values.items():
                                    if size_col in col_name_to_excel_col_idx:
                                        excel_col_idx = col_name_to_excel_col_idx[size_col]
                                        ws.cell(row=excel_row_num, column=excel_col_idx).value = new_val

    # Identify unmatched update items
    df_unmatched_updates = df_updates[~df_updates.index.isin(matched_update_indices)]

    total_new_items_added = 0 # Initialize for new items
    if not df_unmatched_updates.empty:
        print(f"\nFound {len(df_unmatched_updates)} items in the update file that were NOT matched and applied to the original data. Attempting to add them as new rows.")
        display.display(df_unmatched_updates)

        for _, new_row_data in df_unmatched_updates.iterrows():
            style_to_add = pd.to_numeric(new_row_data.get('STYLE #'), errors='coerce')
            po_to_add = pd.to_numeric(new_row_data.get('PO #'), errors='coerce')

            target_ws_for_insert = None
            target_header_idx = None
            target_col_map = {}
            target_col_order = []

            if pd.isna(po_to_add) or pd.isna(style_to_add): # Skip if essential keys are missing
                print(f"Skipping new row with missing PO # or STYLE #: {new_row_data.to_dict()}")
                continue

            if 5000 <= style_to_add <= 5999: # 5000 series, goes to C2
                if c2_ws and c2_header_row_index is not None:
                    target_ws_for_insert = c2_ws
                    target_header_idx = c2_header_row_index
                    target_col_map = c2_col_name_to_excel_col_idx
                    target_col_order = c2_excel_column_order
                else:
                    print(f"Warning: 'C2' sheet not properly identified or processed for new item with STYLE # {style_to_add}. Item skipped: {new_row_data.to_dict()}")
                    continue
            else: # Not 5000 series, goes to Badger
                if badger_ws and badger_header_row_index is not None:
                    target_ws_for_insert = badger_ws
                    target_header_idx = badger_header_row_index
                    target_col_map = badger_col_name_to_excel_col_idx
                    target_col_order = badger_excel_column_order
                else:
                    print(f"Warning: 'Badger' sheet not properly identified or processed for new item with STYLE # {style_to_add}. Item skipped: {new_row_data.to_dict()}")
                    continue

            if not target_ws_for_insert or not target_col_order or not target_col_map:
                print(f"Warning: Insufficient sheet information to add new item. Item skipped: {new_row_data.to_dict()}")
                continue

            # Prepare values to insert, respecting the target sheet's column order
            values_to_insert = []
            for col_name_in_excel_order in target_col_order:
                value = new_row_data.get(col_name_in_excel_order, None)
                # Apply color mapping if it's the COLOR column
                if col_name_in_excel_order == 'COLOR' and value is not None:
                    value = map_compound_color(value, color_mapping)
                values_to_insert.append(value)

            # Find insertion point (based on user's revised logic)
            po_col_idx_in_sheet = target_col_map.get('PO #')
            style_col_idx_in_sheet = target_col_map.get('STYLE #')

            if po_col_idx_in_sheet is None or style_col_idx_in_sheet is None:
                print(f"Warning: 'PO #' or 'STYLE #' column not found in sheet '{target_ws_for_insert.title}'. Cannot sort new item. Appending instead.")
                target_ws_for_insert.append(values_to_insert)
                total_new_items_added += 1
                continue # Skip to next new_row_data

            # Collect existing PO and STYLE numbers with their row indices
            existing_items_in_sheet = []
            for r_num in range(target_header_idx + 2, target_ws_for_insert.max_row + 1):
                existing_po_val = target_ws_for_insert.cell(row=r_num, column=po_col_idx_in_sheet).value
                existing_style_val = target_ws_for_insert.cell(row=r_num, column=style_col_idx_in_sheet).value

                existing_po_num = pd.to_numeric(existing_po_val, errors='coerce')
                existing_style_num = pd.to_numeric(existing_style_val, errors='coerce')

                if pd.notna(existing_po_num) and pd.notna(existing_style_num):
                    existing_items_in_sheet.append({'po': existing_po_num, 'style': existing_style_num, 'row_num': r_num})

            df_existing_items = pd.DataFrame(existing_items_in_sheet)

            insertion_row_index = target_ws_for_insert.max_row + 1 # Default to append to end
            found_insertion_point = False

            if not df_existing_items.empty and po_to_add in df_existing_items['po'].values:
                # Scenario 1: PO exists in the sheet. Insert within its block, sorted by STYLE #.
                po_block_items = df_existing_items[df_existing_items['po'] == po_to_add].sort_values(by='style')

                # Find the insertion point within this PO's block
                for idx, row_data in po_block_items.iterrows():
                    if style_to_add < row_data['style']:
                        insertion_row_index = row_data['row_num']
                        found_insertion_point = True
                        break

                if not found_insertion_point:
                    # If new style is largest in the block, insert after the last row of the block
                    insertion_row_index = po_block_items['row_num'].max() + 1
                    found_insertion_point = True
            else:
                # Scenario 2: PO does NOT exist in the sheet. Add at the top (after header row).
                # This means it will be inserted as the first data row.
                insertion_row_index = target_header_idx + 2
                found_insertion_point = True

            # Insert a new blank row
            target_ws_for_insert.insert_rows(insertion_row_index)

            # Determine the source row for copying formatting.
            source_row_for_formatting = None

            # Try to copy from the row that was just shifted down (now at insertion_row_index + 1)
            # Use iter_rows to get the row to avoid potential __getitem__ issues.
            if insertion_row_index + 1 <= target_ws_for_insert.max_row:
                for row_tuple in target_ws_for_insert.iter_rows(min_row=insertion_row_index + 1, max_row=insertion_row_index + 1):
                    source_row_for_formatting = row_tuple
                    break
            # If inserting as the first data row and there's a data row below it, copy from that one
            elif insertion_row_index == target_header_idx + 2 and target_ws_for_insert.max_row > target_header_idx + 2:
                for row_tuple in target_ws_for_insert.iter_rows(min_row=target_header_idx + 2, max_row=target_header_idx + 2):
                    source_row_for_formatting = row_tuple
                    break
            # If appending and there's a row above, copy from the one above
            elif insertion_row_index > target_header_idx + 2 and insertion_row_index - 1 > target_header_idx:
                for row_tuple in target_ws_for_insert.iter_rows(min_row=insertion_row_index - 1, max_row=insertion_row_index - 1):
                    source_row_for_formatting = row_tuple
                    break

            # Apply formatting and then write values
            for col_name_from_order, value in zip(target_col_order, values_to_insert):
                excel_col_to_write_to = target_col_map.get(col_name_from_order)
                if excel_col_to_write_to is not None:
                    new_cell = target_ws_for_insert.cell(row=insertion_row_index, column=excel_col_to_write_to)
                    new_cell.value = value

                    if source_row_for_formatting:
                        source_cell_idx = excel_col_to_write_to - 1 # openpyxl cell access by column index in a row is 0-based
                        if source_cell_idx < len(source_row_for_formatting):
                            source_cell = source_row_for_formatting[source_cell_idx]
                            new_cell.font = source_cell.font.copy()
                            new_cell.border = source_cell.border.copy()
                            new_cell.fill = source_cell.fill.copy()
                            new_cell.number_format = source_cell.number_format
                            new_cell.protection = source_cell.protection.copy()
                            new_cell.alignment = source_cell.alignment.copy()

            total_new_items_added += 1

        if total_new_items_added > 0:
            print(f"Added {total_new_items_added} new items to respective sheets.")


    # Final save and download logic, combining both updates and new additions
    if total_updates_applied > 0 or total_new_items_added > 0: # `total_new_items_added` only increases if not .xls
        print(f"\nTotal {total_updates_applied} unique update requests processed and {total_new_items_added} new items added from the update file.")
        print(f"Formatted updated data is being saved to '{output_file_name_formatted}'.")
        workbook.save(output_file_name_formatted)
        print(f"Formatted updated data saved to '{output_file_name_formatted}'.")
        from google.colab import files
        files.download(output_file_name_formatted)
    else: # If no updates or new items were processed for the .xlsx file
        print("\nNo updates were applied and no new items were added to the Excel file.")

print("\nProcess Complete.")

Please upload your main Excel file (e.g., 'Container Plan & WIP.xlsx').


Saving Container Plan & WIP After F-575.xls to Container Plan & WIP After F-575 (3).xls
Successfully uploaded 'Container Plan & WIP After F-575 (3).xls'. Now attempting to extract 'COLOR' column from Sheets.

--- Now, please upload the Excel or CSV file containing your updates. ---
It should have 'PO #', 'STYLE #', 'COLOR' columns and all size columns (XS, S, M, L, XL, 2XL, 3XL, 4XL).


/tmp/ipykernel_2015/3695464257.py:182: FutureWarning: 'any' with datetime64 dtypes is deprecated and will raise in a future version. Use (obj != pd.Timestamp(0)).any() instead.
  if (df_sheet.any() == "GRAND TOTAL").any().any():
/tmp/ipykernel_2015/3695464257.py:182: FutureWarning: 'any' with datetime64 dtypes is deprecated and will raise in a future version. Use (obj != pd.Timestamp(0)).any() instead.
  if (df_sheet.any() == "GRAND TOTAL").any().any():
/tmp/ipykernel_2015/3695464257.py:182: FutureWarning: 'any' with datetime64 dtypes is deprecated and will raise in a future version. Use (obj != pd.Timestamp(0)).any() instead.
  if (df_sheet.any() == "GRAND TOTAL").any().any():


Saving pivot_with_totals (10).xlsx to pivot_with_totals (10) (3).xlsx
Successfully loaded update file 'pivot_with_totals (10) (3).xlsx'. First 5 rows:


,ORD #,PO #,DUE DATE,STYLE #,COLOR,XS,S,M,L,XL,2XL,3XL,4XL,TOTAL
0,NaN,1735988.0,NaN,5500,OXFORD,0,0,0,0,24,0,0,0,24
1,NaN,1735988.0,NaN,5500,WHITE,0,24,0,0,0,0,0,0,24
2,NaN,1735988.0,NaN,5501,CHARCOAL,24,0,0,0,0,0,0,0,24
3,NaN,1735988.0,NaN,5501,FOREST,0,0,0,0,0,0,12,24,36
4,NaN,1735988.0,NaN,5520,OXFORD,24,0,0,0,0,0,0,0,24



--- Debugging: Standardized Column Names ---
df_combined columns: ['ORD #', 'PO #', 'DUE DATE', 'STYLE #', 'COLOR', 'XS', 'S', 'M', 'L', 'XL', '2XL', '3XL', '4XL', 'TOTAL', 'ESTIMATED DATE', 'UNNAMED: 15', 'F-576 - 06/10', 'F-577 - 6/25', 'F-578 - 07/10', 'F-579 - 7/20', 'F-580 - 07/30', 'F-581 - 8/10', 'F-575 - 05/18', 'F-576 - 05/30', 'F-577 - 10/06', 'F-578 - 06/20', 'F-579 - 06/30', 'F-580 - 10/07', 'F-581 - 20/07']
df_updates columns: ['ORD #', 'PO #', 'DUE DATE', 'STYLE #', 'COLOR', 'XS', 'S', 'M', 'L', 'XL', '2XL', '3XL', '4XL', 'TOTAL']
Error: openpyxl does not support the old .xls file format for preserving formatting.
Please convert your main Excel file to the .xlsx format and re-upload it to use the formatting preservation feature.

Process Complete.


In [ ]:
from google.colab import files
files.download(output_file_name_formatted)

FileNotFoundError: Cannot find file: Updated_Container_Plan_Formatted.xlsx

In [ ]:
print('--- Items from the update file that were successfully matched and updated ---')
updated_items_df = df_updates.loc[matched_update_indices]
display.display(updated_items_df)


--- Items from the update file that were successfully matched and updated ---


,PO #,STYLE #,COLOR,XS,S,M,L,XL,2XL,3XL,4XL
2,1735988,5501,CHARCOAL,24,0,0,0,0,0,0,0
3,1735988,5501,FOREST,0,0,0,0,0,0,12,24
4,1735988,5501,RED,24,0,0,0,0,0,0,0
5,1735988,5520,OXFORD,24,0,0,0,0,0,0,0
6,1735988,5521,FOREST,24,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
332,1738643,5577,OXFORD,144,1632,2376,1488,384,144,0,0
333,1738643,5578,BLACK,24,0,0,0,0,48,0,0
334,1738643,5578,CHARCOAL,0,24,0,0,0,0,0,0
335,1738643,5578,NAVY,24,72,48,48,24,0,0,0


In [ ]:
# This cell is a duplicate of yAxP6PFpkk9m and can be ignored or deleted. The main processing code is in cell yAxP6PFpkk9m.

In [ ]:
identity_cols = ['PO #', 'STYLE #', 'COLOR']

# Get the list of all columns that are in df_combined (this will be the superset of all possible columns)
all_data_cols = df_combined.columns.tolist()

# Get the unique keys from the updated_items_df (updates that were actually applied)
updated_keys_df = updated_items_df[identity_cols].drop_duplicates()

# Dictionary to store comparison results for each sheet
sheet_comparisons = {}

# Create a new Excel writer for the output
output_excel_comparison_file = 'updated_items_comparison_sheet_wise.xlsx'
output_excel_writer = pd.ExcelWriter(output_excel_comparison_file, engine='xlsxwriter')

for descriptive_sheet_name, original_df_for_sheet in all_sheets_data.items():
    print(f"\n--- Processing comparison for: {descriptive_sheet_name} ---")

    # Filter the original sheet data to only include items that were targeted by updates
    # This ensures we only look at rows from this specific sheet that match an update key
    sheet_relevant_keys = original_df_for_sheet[identity_cols].merge(updated_keys_df, on=identity_cols, how='inner').drop_duplicates()

    if sheet_relevant_keys.empty:
        print(f"No relevant updated items found in {descriptive_sheet_name}.")
        continue

    # Filter df_combined_before_update and df_combined to get rows specific to this sheet and these update keys
    # Merge on identity_cols to keep only the rows that exist in both the sheet's original data and the updated keys
    df_before_sheet_filtered = pd.merge(
        df_combined_before_update,
        sheet_relevant_keys,
        on=identity_cols,
        how='inner'
    )
    df_after_sheet_filtered = pd.merge(
        df_combined,
        sheet_relevant_keys,
        on=identity_cols,
        how='inner'
    )

    if df_before_sheet_filtered.empty or df_after_sheet_filtered.empty:
        print(f"No 'before' or 'after' data found for relevant updated items in {descriptive_sheet_name}.")
        continue

    # Set identity columns as index for easier comparison
    df_before_sheet_filtered = df_before_sheet_filtered.set_index(identity_cols)
    df_after_sheet_filtered = df_after_sheet_filtered.set_index(identity_cols)

    # Ensure dtypes are consistent before comparison by converting to string, especially for mixed types
    # Using .fillna('') to treat NaNs as empty strings for robust comparison
    current_sheet_cols = list(set(df_before_sheet_filtered.columns) & set(df_after_sheet_filtered.columns))

    df_before_compare = df_before_sheet_filtered[current_sheet_cols].astype(str).fillna('').sort_index(axis=1)
    df_after_compare = df_after_sheet_filtered[current_sheet_cols].astype(str).fillna('').sort_index(axis=1)

    # Find differences (where 'before' is not equal to 'after')
    differences = (df_before_compare != df_after_compare)

    # Get rows where there is at least one difference in any column
    diff_rows_indices = differences.any(axis=1)

    if diff_rows_indices.any():
        print(f'--- Before and After State of Updated Items with Changed Values for {descriptive_sheet_name} ---')

        # Filter the original indexed dataframes to only include rows with differences
        df_before_with_diffs = df_before_sheet_filtered.loc[diff_rows_indices]
        df_after_with_diffs = df_after_sheet_filtered.loc[diff_rows_indices]

        # Initialize display_df with the same index as the filtered dataframes
        display_df = pd.DataFrame(index=df_before_with_diffs.index)

        # Add identity columns to display_df directly from the index (they are consistent)
        for id_col in identity_cols:
            display_df[id_col] = display_df.index.get_level_values(id_col)

        # Iterate through all columns that could potentially show a change, preserving original order if possible
        # Use `current_sheet_cols` which are common to both before and after, for this specific sheet
        cols_for_display_order = [col for col in current_sheet_cols if col not in identity_cols]

        for col in cols_for_display_order:
            # Check if this specific column had a change in any of the *filtered* rows
            if col in differences.columns and differences.loc[diff_rows_indices, col].any():
                display_df[f'{col}_BEFORE'] = df_before_with_diffs[col]
                display_df[f'{col}_AFTER'] = df_after_with_diffs[col]

        # Reset index for a clean display of the final DataFrame
        df_actual_changes_sheet = display_df.reset_index(drop=True)
        display.display(df_actual_changes_sheet)
        sheet_comparisons[descriptive_sheet_name] = df_actual_changes_sheet

        # Write to Excel sheet
        df_actual_changes_sheet.to_excel(output_excel_writer, sheet_name=descriptive_sheet_name, index=False)

    else:
        print(f'No value changes were detected in the updated rows for {descriptive_sheet_name}.')

# Save the Excel file with all comparison sheets
if sheet_comparisons:
    output_excel_writer.close()
    print(f'\nComparison reports saved sheet-wise to {output_excel_comparison_file}')
    files.download(output_excel_comparison_file)
else:
    print('\nNo changes found across any sheets for comparison.')


--- Processing comparison for: Sheet 2 ---
--- Before and After State of Updated Items with Changed Values for Sheet 2 ---


,PO #,STYLE #,COLOR,XL_BEFORE,XL_AFTER,L_BEFORE,L_AFTER,M_BEFORE,M_AFTER,2XL_BEFORE,2XL_AFTER,4XL_BEFORE,4XL_AFTER,S_BEFORE,S_AFTER,3XL_BEFORE,3XL_AFTER,XS_BEFORE,XS_AFTER
0,10006914.0,4980.0,ROYAL/OXFORD/,NaN,0.0,NaN,0.0,72.0,72.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
1,10006914.0,4981.0,ROYAL/OXFORD/,NaN,0.0,36.0,36.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
2,10006914.0,1286.0,NAVY,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,12.0,12.0,NaN,0.0,NaN,0.0,NaN,0.0
3,10006946.0,1286.0,NAVY,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,12.0,12.0,NaN,0.0
4,10007025.0,1286.0,BLACK,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,1738643.0,2254.0,OXFORD,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0,NaN,0.0,24.0,24.0
203,1738643.0,2277.0,BLACK,NaN,0.0,336.0,336.0,240.0,240.0,NaN,0.0,NaN,0.0,168.0,168.0,NaN,0.0,NaN,0.0
204,1738643.0,2277.0,NAVY,NaN,0.0,144.0,144.0,120.0,120.0,NaN,0.0,NaN,0.0,72.0,72.0,NaN,0.0,24.0,24.0
205,1738643.0,2277.0,CHARCOAL,NaN,0.0,120.0,120.0,96.0,96.0,NaN,0.0,NaN,0.0,48.0,48.0,NaN,0.0,NaN,0.0



--- Processing comparison for: Sheet 3 ---
--- Before and After State of Updated Items with Changed Values for Sheet 3 ---


,PO #,STYLE #,COLOR,XL_BEFORE,XL_AFTER,L_BEFORE,L_AFTER,M_BEFORE,M_AFTER,2XL_BEFORE,2XL_AFTER,4XL_BEFORE,4XL_AFTER,S_BEFORE,S_AFTER,3XL_BEFORE,3XL_AFTER,XS_BEFORE,XS_AFTER
0,1735988.0,5501.0,FOREST,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0,NaN,0.0,12.0,12.0,NaN,0.0
1,1735988.0,5501.0,RED,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0
2,1735988.0,5501.0,CHARCOAL,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0
3,1735988.0,5520.0,OXFORD,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0
4,1735988.0,5521.0,FOREST,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1738643.0,5577.0,OXFORD,384.0,384.0,1488.0,1488.0,2376.0,2376.0,144.0,144.0,NaN,0.0,1632.0,1632.0,NaN,0.0,144.0,144.0
60,1738643.0,5578.0,BLACK,NaN,0.0,NaN,0.0,NaN,0.0,48.0,48.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0
61,1738643.0,5578.0,NAVY,24.0,24.0,48.0,48.0,48.0,48.0,NaN,0.0,NaN,0.0,72.0,72.0,NaN,0.0,24.0,24.0
62,1738643.0,5578.0,CHARCOAL,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,24.0,24.0,NaN,0.0,NaN,0.0



--- Processing comparison for: Sheet 4 ---
No relevant updated items found in Sheet 4.

Comparison reports saved sheet-wise to updated_items_comparison_sheet_wise.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# This cell was removed as the comparison is now handled sheet-wise and downloaded in the previous cell.